In [ ]:
from transformers import BertTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments, BertForMaskedLM, AutoTokenizer, EarlyStoppingCallback
from datasets import load_dataset
import datasets
import torch
import torch.optim as optim
import re
from nltk.tokenize import sent_tokenize
import nltk
import accelerate
from google_cloud_save import gcs_get_dataset, upload_folder
import os

nltk.download('punkt_tab')


In [ ]:
#TODO put all editable hyperparameters in one block up here for ease of editing
#also tune the parameters based on the DAPT paper
#save models based on steps not epochs, save every 500ish steps
#raise batch size to 32-64
#warm up ratio for adam optimizer
#weight decay for AdamW
#record hyperparameters when saving final model
#early stopping when model stops improving


#TODO make sure we're saving logging_dir to the right place so it gets uploaded




In [ ]:
#HYPERPARAMETERS FOR TRAINING

MODEL_NAME = "100-Sample-Pretrained-Model-3"

epochs = 3
learning_rate = 0.005
batch_size = 32
logging_steps = 100
warmup_ratio = 0.05
weight_decay = 0.01
save_steps = 500
save_strategy = 'steps'
eval_strategy = 'epochs'
best_model_metric = 'eval_loss'
callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]

mlm_probability = 0.15





In [ ]:
#Load dataset and choose the model we want to use
#TODO Replace gcs_credentials path with your own service account information json file, contact Ryan for help.
gcs_credentaials = "nlp-research-sp26-8499634f1c62.json"

#load the dataset into a hugginface datasets object
dataset = gcs_get_dataset(credentials_path=gcs_credentaials, bucket_name="project3102-data-bucket", data_blob_path="sample_1950s_1960s.jsonl")

model_name = "bert-base-uncased"


In [ ]:
dataset[:1]

In [ ]:
# get all dates from dataset, this function currently does years, will need to update to decades
def get_date_tokens(dataset: datasets.Dataset):
    unique_dates = list(set(sorted(dataset['decade'])))
    unique_dates = [str(d).removesuffix('s') for d in unique_dates]
    custom_date_tokens = [f"<decade_{d}>" for d in unique_dates]
    return custom_date_tokens

In [ ]:
#create the tokenizer and add custom tokens
#extra_special_tokens tag is for any non-standard special tokens so we'll use it for all the dates

date_tokens = get_date_tokens(dataset)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.add_special_tokens({'extra_special_tokens' : date_tokens})


In [ ]:
#append date tokens to the start of all sentences and create simplified dataset

#1. create a dict to hold samples
sentenceData = {'text': []}
#2. iterate entries
for entry in dataset:
    text = entry['text'] # type: ignore
    date = entry['decade'] # type: ignore
    date = str(date).removesuffix('s')
    #3. split entry into sentences.
    for sentence in sent_tokenize(text):
        #4. append date token, limit sentence length to the bert maximum input size, and add to new dataset
        sentence = f'<year_{date}> '+ sentence
        sentenceData['text'].append(sentence)

    #TODO combine sentences until we reach near max length IN TOKENS NOT CHARS
    #that will give us chunks instead of single sentences
#5. create cleaned dataset from sample dict
timestamped_dataset = datasets.Dataset.from_dict(sentenceData)


In [ ]:
#tokenizer function used to map cleaned samples to tokenizer token ids
def tokenize_data(examples):
    result = tokenizer(examples["text"], max_length=512)
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result



In [ ]:
# Create Data collator for Masked Language Modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=mlm_probability)

# split dataset into train and validation sets
split = timestamped_dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split['train']
val_dataset   = split['test']

print(f"Train samples: {len(train_dataset)}")
print(f"Val   samples: {len(val_dataset)}")

# tokenize datasets
train_dataset = train_dataset.map(tokenize_data, batch_size=batch_size, batched=True)
val_dataset   = val_dataset.map(tokenize_data,   batch_size=batch_size, batched=True)

In [ ]:
# Load pre-trained model and resize to the custom tokenizer
model = BertForMaskedLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

In [ ]:
#set model name and output directory
output_dir = f"Models/{model_name}"
optimizer = optim.AdamW(params = model.parameters(), lr = learning_rate, weight_decay = weight_decay)


#Train using 
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,

    #saving model based on eval_loss
    eval_strategy=eval_strategy,
    save_strategy=save_strategy,
    load_best_model_at_end=True,
    metric_for_best_model=best_model_metric,
    greater_is_better=False,
    save_total_limit=2,
    logging_dir=os.path.join(output_dir, '/logs'),
    logging_steps=logging_steps,
    warmup_ratio= warmup_ratio
)


# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    optimizer=optimizer
)

# Pretrain the model
trainer.train()


#save model
save_path = f"{output_dir}/best"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)


In [ ]:
#model info saving

import json


train_results = trainer.state.log_history

lr = training_args.learning_rate                       
warmup_ratio = training_args.warmup_ratio               
weight_decay = training_args.weight_decay    

data = {
    "model_name": model_name,
    "epoch": training_args.num_train_epochs,
    "batch_size": training_args.per_device_train_batch_size,
    "learning_rate": lr,
    "logging_steps": training_args.logging_steps,
    "warmup_ratio": warmup_ratio,
    "weight_decay": weight_decay,
    "training_loss": training_loss,
    "eval_loss": eval_loss,
}


# Save to file
save_path = f"{output_dir}/parameters.json"
with open(save_path, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Saved parameters to {json_save_path}")

In [ ]:
#upload model to GCS
local_dir = save_path
bucket_name = "project3102-model-bucket"
destination_blob_prefix = f"Training-Tests/{model_name}/" # Folder path in GCS

upload_folder(credentials_path=gcs_credentaials, bucket_name=bucket_name, destination_blob_prefix=destination_blob_prefix, local_dir=local_dir)